In [ ]:
!pip install emoji
!pip install vncorenlp
!pip install pyahocorasick

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 55.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for vncorenlp: filename=vncorenlp-1.0.3-py3-none-any.whl size=2645933 sha256=d5065d3d29ae10955d6ef029a074953cb01846438228ec67af3d23f71d3e3977
  Stored in directory: /root/.cache/pip/wheels/6f/19/20/ec7083125fd06db1a19d0d3ca18806ecf4e8ed1464713b4efa
Successfully built vncorenlp
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import emoji

In [ ]:
df=pd.read_csv("YOUTUBE_GOC.csv")
df.head()

,video_id,title,hashtags,channel,channel_id,subscribers,views,likes,comments,category,published_at,duration,top_comments
0,vmaKtJuJaK0,Ronaldo Không Muốn Làm Sạch Tai,#shorts,Cà Rô Naldo,UCPZrnJyg-sQTozu9ZxgOoZQ,336000,1834534,60499,89,Sports,2025-10-30T09:01:27Z,28,@@MinhDuy-e9w2c: Mua mọe cái áo thun mcraft bê...
1,lEhUBSyK41g,TRỌNG TÀI BUỘC PHẢI VI PHẠM LUẬT BÓNG ĐÁ 😳 | @...,NaN,Date With Gym,UCHCbPLCd3nEAlNl2C55rsJg,424000,2923222,35918,109,Sports,2025-10-25T00:01:05Z,19,@@tienatvu701: Khán giat phản đối vì họ đaetj ...
2,Zz4rP6nVqos,Thể thao biến thành phép màu | Rì Còn Viu,#riconviu,Rì Còn Viu,UC8my4vFHCKY1ExaG99zjuIA,131000,1057566,21308,27,Sports,2025-10-29T03:00:46Z,49,@@Noob_sadboy-ok3pe: Chỉ dàng dc huy chương và...
3,Dq3byX1poFc,Pha giao bóng đi vào lịch sử,"#familyguy,#longtieng,#fgvn99",Người Đàn Ông Của Gia Đình,UCzI57speLHzZgl2Qq-cqe9Q,350000,1230500,59361,471,Sports,2025-10-27T11:00:22Z,60,@@QuangKhaiKien: hehe | @@DũngTrí-u7ol: 11/9💀 ...
4,w5Pbb_iPCDE,Cô bé có tốc độ siêu đỉnh,#shorts,Nư Nư,UCECp_KHVNi8So3RcmSbyzhw,490000,333699,3660,3,Sports,2025-10-31T12:32:53Z,14,@@ChuyênĐậu: Ngu | @@Liên-f5y: ❤❤❤🎉🎉🎉 | @@thil...


In [ ]:
df.isna().sum()

,0
video_id,0
title,10
hashtags,291
channel,0
channel_id,0
subscribers,0
views,0
likes,0
comments,0
category,0


Dữ liệu có 10 dữ liệu thiếu ở cột title và 291 dữ liệu thiếu ở cột hashtags, 12 dữ liệu thiếu ở cột top_comments

# tiền xử lý dữ liệu

In [ ]:
df.describe()

,subscribers,views,likes,comments,duration
count,5.000000e+02,5.000000e+02,5.000000e+02,500.000000,500.00000
mean,2.444948e+06,3.049101e+06,5.568490e+04,760.894000,1209.87200
std,2.056606e+07,7.136084e+06,1.410339e+05,3688.669409,4502.74629
min,1.110000e+04,9.755000e+03,0.000000e+00,0.000000,10.00000
25%,8.705000e+04,3.544455e+05,4.068750e+03,28.000000,24.00000
50%,3.420000e+05,9.104920e+05,1.231500e+04,93.000000,44.00000
75%,1.145000e+06,2.306721e+06,3.347900e+04,326.000000,244.25000
max,4.490000e+08,5.601212e+07,1.581630e+06,56913.000000,39309.00000


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   video_id      500 non-null    object
 1   title         490 non-null    object
 2   hashtags      209 non-null    object
 3   channel       500 non-null    object
 4   channel_id    500 non-null    object
 5   subscribers   500 non-null    int64 
 6   views         500 non-null    int64 
 7   likes         500 non-null    int64 
 8   comments      500 non-null    int64 
 9   category      500 non-null    object
 10  published_at  500 non-null    object
 11  duration      500 non-null    int64 
 12  top_comments  488 non-null    object
dtypes: int64(5), object(8)
memory usage: 50.9+ KB


Dữ liệu ban đầu chứa 500 bản ghi là thông tin của 500 video đang trên top trending youtube Việt Nam, được lấy ngày 1/11/2025. Chứa 13 trường dữ liệu về mã video, tên, hashtag, kênh đăng tải, mã kênh đăng tải, số lượng theo dõi kênh, số lượt xem, số lượt thích, số lượt bình luận, thể loại video, thời gian đăng tải, thời lượng video và top 50 bình luận nổi bật nhất của từng video.

In [ ]:
# Xử lý
df['hashtags'] = df['hashtags'].fillna('No hashtag')
df['title']=df['title'].fillna('No title')
df['top_comments']=df['top_comments'].fillna('No comment')

In [ ]:
df.isna().sum()

,0
video_id,0
title,0
hashtags,0
channel,0
channel_id,0
subscribers,0
views,0
likes,0
comments,0
category,0


Thay giá trị thiếu ở cột 'hashtags' bằng "No hashtags", thay giá trị thiếu ở cột 'title' bằng "No title".

In [ ]:
# kiểm tra dữ liệu trùng lặp
df.duplicated().any()

np.False_

không có dữ liệu bị trùng lặp

In [ ]:
# chuẩn hóa và làm sạch văn bản
df['title'] = df['title'].str.strip().str.lower()
df['hashtags'] = df['hashtags'].str.strip().str.lower()
df['channel']=df['channel'].str.strip().str.lower()
df['category']=df['category'].str.strip().str.lower()
df['top_comments']=df['top_comments'].str.strip().str.lower()

chuẩn hóa các cột văn bản để dễ xử lý ngôn ngữ bằng cách loại bỏ ký tự đặc biệt, khoảng trắng dư, viết thường toàn bộ.

In [ ]:
# lọc bot/spam
df = df[df['subscribers'] > 10]
df = df[df['views'] > 0]
df = df[df['likes'] > 0]
df = df[df['comments'] > 0]

Loại bỏ các video có số lượng đăng ký kênh ít hơn 10, lượng views, likes, comments = 0

In [ ]:
# chuẩn hóa cột thời gian
df['published_at'] = pd.to_datetime(df['published_at'], utc=True)
df['published_at'] = df['published_at'].dt.tz_localize(None)

In [ ]:
# tạo cột tỉ lệ tương tác
df['engagement_rate'] = (df['likes'] + df['comments']) / df['views']

In [ ]:
def clean_text(text):
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^a-zàáảãạăắằẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọ'
                  r'ôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_comments(text):
    items = text.split(' | ')
    comments = [item.split(': ', 1)[1] for item in items if ': ' in item]
    return ' | '.join(comments)

def tokenize_hashtags(text):
    if pd.isna(text):
        return []
    s = str(text).strip()
    if s.lower() in ('no hashtag', 'no hashtags', ''):
        return []
    parts = re.split(r'[,\s]+', s)
    parts = [p.lstrip('#').lower() for p in parts if p.strip()]
    return parts

df['cleaned_title'] = df['title'].apply(clean_text)
df['top_comments'] = df['top_comments'].apply(extract_comments)
df['hashtags_tokens'] = df['hashtags'].apply(tokenize_hashtags)


In [ ]:
df.head()

,video_id,title,hashtags,channel,channel_id,subscribers,views,likes,comments,category,published_at,duration,top_comments,engagement_rate,cleaned_title,hashtags_tokens
0,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,#shorts,cà rô naldo,UCPZrnJyg-sQTozu9ZxgOoZQ,336000,1834534,60499,89,sports,2025-10-30 09:01:27,28,mua mọe cái áo thun mcraft bên ngoài cón hơn |...,0.033026,ronaldo không muốn làm sạch tai,[shorts]
1,lEhUBSyK41g,trọng tài buộc phải vi phạm luật bóng đá 😳 | @...,no hashtag,date with gym,UCHCbPLCd3nEAlNl2C55rsJg,424000,2923222,35918,109,sports,2025-10-25 00:01:05,19,khán giat phản đối vì họ đaetj cho đội kia :))...,0.012324,trọng tài buộc phải vi phạm luật bóng đá tt da...,[]
2,Zz4rP6nVqos,thể thao biến thành phép màu | rì còn viu,#riconviu,rì còn viu,UC8my4vFHCKY1ExaG99zjuIA,131000,1057566,21308,27,sports,2025-10-29 03:00:46,49,chỉ dàng dc huy chương vàng ??? | cuối cùng 2 ...,0.020174,thể thao biến thành phép màu rì còn viu,[riconviu]
3,Dq3byX1poFc,pha giao bóng đi vào lịch sử,"#familyguy,#longtieng,#fgvn99",người đàn ông của gia đình,UCzI57speLHzZgl2Qq-cqe9Q,350000,1230500,59361,471,sports,2025-10-27 11:00:22,60,hehe | 11/9💀 | 😂 | tôi k hiểu cái đoạn mấy tay...,0.048624,pha giao bóng đi vào lịch sử,"[familyguy, longtieng, fgvn99]"
4,w5Pbb_iPCDE,cô bé có tốc độ siêu đỉnh,#shorts,nư nư,UCECp_KHVNi8So3RcmSbyzhw,490000,333699,3660,3,sports,2025-10-31 12:32:53,14,ngu | ❤❤❤🎉🎉🎉 | em tuyệt vời ❤❤❤,0.010977,cô bé có tốc độ siêu đỉnh,[shorts]


In [ ]:
output_path = "diu_tu_be.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 471 entries, 0 to 499
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   video_id         471 non-null    object        
 1   title            471 non-null    object        
 2   hashtags         471 non-null    object        
 3   channel          471 non-null    object        
 4   channel_id       471 non-null    object        
 5   subscribers      471 non-null    int64         
 6   views            471 non-null    int64         
 7   likes            471 non-null    int64         
 8   comments         471 non-null    int64         
 9   category         471 non-null    object        
 10  published_at     471 non-null    datetime64[ns]
 11  duration         471 non-null    int64         
 12  top_comments     471 non-null    object        
 13  engagement_rate  471 non-null    float64       
 14  cleaned_title    471 non-null    object        

sau khi xử lý, dữ liệu còn lại 471 bản ghi của 15 trường dữ liệu, xuất hiện thêm trường engagement_rate và cleaned_title. 29 bản ghi đã được loại bỏ, đảm bảo tính chính xác và thực tế cho dữ liệu.

In [ ]:
def expand_comments(df):
    # Chọn các cột cần dùng
    df = df[['video_id', 'cleaned_title', 'category', 'top_comments']].copy()
    # Tách top_comments thành list
    df['top_comments'] = df['top_comments'].str.split('|')
    # Dùng explode để mỗi comment thành 1 dòng mới
    df_expanded = df.explode('top_comments')
    # Làm sạch comment
    df_expanded['top_comments'] = df_expanded['top_comments'].str.strip()
    df_expanded = df_expanded[['video_id', 'cleaned_title', 'category', 'top_comments']]
    # Reset index
    df_expanded = df_expanded.reset_index(drop=True)
    return df_expanded

df_expanded = expand_comments(df)
df_expanded.head()

,video_id,cleaned_title,category,top_comments
0,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,mua mọe cái áo thun mcraft bên ngoài cón hơn
1,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,❤❤
2,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,❤❤❤😊
3,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,***
4,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,đi việt nam đi


In [ ]:
def count_emojis(text):
    return sum(1 for char in text if char in emoji.EMOJI_DATA)

emoji_to_vietnamese = {
    '😂': 'ha ha',
    '🤣': 'ha ha',
    '😍': 'yêu thích',
    '😘': 'yêu thích',
    '😙': 'yêu thích',
    '😚': 'yêu thích',
    '👍': 'thích',
    '👎': 'không thích',
    '😢': 'hu hu',
    '😡': 'giận',
    '😱': 'sốc',
    '💔': 'tan vỡ',
    '❤': 'yêu thích',
    '😃': 'ha ha',
    '😊': 'ha ha',
    '😭': 'hu hu',
    '🔥': 'tuyệt vời',
    '🎉': 'tuyệt vời',
    '😞': 'hu hu',
    '😨': 'sợ',
    '👌': 'oke',
    '❓': '?',
    '😉': 'oke',
    '💯': 'tuyệt vời',
    '😮': 'wow',
    '😒': 'chịu',
    '✅️': 'đúng',
    '❌️': 'sai',
    '❤️': 'yêu thích',
    '😅': 'ha ha'
}

def replace_emoji(text):

    for emot, translation in emoji_to_vietnamese.items():
        text = text.replace(emot, f" {translation} ")
    return text.strip()

# Regex bắt tất cả emoji trong chuỗi
emoji_regex = re.compile("|".join(map(re.escape, emoji_to_vietnamese.keys())))

def clean_emoji_text(text):
    # Thay emoji hợp lệ
    text = emoji_regex.sub(lambda m: " " + emoji_to_vietnamese[m.group()] + " ", text)
    # Loại bỏ emoji KHÔNG nằm trong danh sách
    text = re.sub(r'[^\w\s,.!?áàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđÁÀẢÃẠĂẮẰẲẴẶÂẤẦẨẪẬÉÈẺẼẸÊẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÚÙỦŨỤƯỨỪỬỮỰÝỲỶỸỴĐ ]', '', text)
    # Chuẩn hóa khoảng trắng
    return re.sub(r'\s+', ' ', text).strip()

# Define slang and their polite substitutions
slang_to_polite = {
    "dm": "trời ơi",
    'tao': 'tôi',
    'đéo': 'không',
    "vãi": "quá",
    "đm": "trời ơi",
    'đcm': 'trời ơi',
    'đcmm': 'trời ơi',
    "vcl": "quá",
    "vl": "quá",
    "clm": "trời đất ơi",
    "cmn": "thật là",
    "cặc": "khó chịu",
    "địt": "khó chịu",
    "lồn": "khó chịu",
    "lol": "ha ha",
    "wtf": "gì vậy trời",
    "ck": "chồng",
    "trc": "trước",
    "vs": "với",
    "mn": "mọi người",
    "ng": "người",
    "dc": "được",
    "k": "không",
    'ko': 'không',
    "hok": "không",
    'cmnl': 'luôn',
    "ntn": "như thế nào",
    "nt": "nhắn tin",
    "admin": "quản trị viên",
    "fb": "facebook",
    "ig": "instagram",
    "yt": "youtube",
    'ae': 'anh em',
    'hp': 'hạnh phúc',
    'sn': 'sinh nhật',
    'ce': 'chị em',
    "j": "gì",
    'cón': 'còn',
    'thik': 'thích',
    'thks': 'cảm ơn',
    'tks': 'cảm ơn',
    'mk': 'mình',
    'p': 'phải',
    'kq': 'kết quả',
    'ns': 'nói',
    'rl': 'thật',
    'stt': 'status',
    'tb': 'thông báo',
    'tbh': 'nói thật',
    'mcraft': 'minecraft',
    'dcq': 'được quái',
    'đcq': 'được quái',
    'thg': 'thằng',
    'v': 'vậy',
    'dc': 'được',
    'đc': 'được',
    'rê': 'ghê',
    'r': 'rồi',
    'đt': 'điện thoại',
    'mnz': 'mọi người',
    'bro': 'anh em',
    'sis': 'chị em',
    'đù': 'ôi trời',
    'đú': 'bắt chước',
    'hé': 'nè',
    'héy': 'nè',
    'hix': 'hu hu',
    'híc': 'hu hu',
    'piz': 'làm ơn',
    'plz': 'làm ơn',
    'thx': 'cảm ơn',
    'ty': 'cảm ơn',
    'cmt': 'bình luận',
    'rep': 'trả lời',
    'sub': 'đăng ký',
    'subs': 'đăng ký',
    'ads': 'quảng cáo',
    'ad': 'quảng cáo',
    'si': 'sĩ',
    't': 'tôi',
    'u': 'bạn',
    'm': 'bạn',
    'e': 'em',
    'a': 'anh',
    'ur': 'của bạn',
    'lun': 'luôn',
    'qúa': 'quá',
    'kím': 'kiếm',
    'qué': 'quá',
    'hlv': 'huấn luyện viên',
    'mọe': 'luôn'
}

# Function to replace slang terms in a comment
def replace_slang(comment, mapping):
    pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in mapping.keys()) + r')\b', flags=re.IGNORECASE)
    modified_comment = pattern.sub(lambda x: mapping.get(x.group().lower(), x.group()), comment)
    return modified_comment

stopwords = [
    'a', 'à', 'á', 'ả', 'ã', 'ạ', 'anh', 'chị', 'em', 'tôi', 'ta', 'mày', 'mình',
    'cô', 'chú', 'bác', 'ông', 'bà', 'các', 'những', 'cái', 'đó', 'này', 'kia',
    'vậy', 'thế', 'đây', 'đấy', 'vừa', 'mới', 'đã', 'rồi', 'sẽ', 'và', 'hoặc',
    'hay', 'nhưng', 'vẫn', 'thì', 'là', 'ở', 'đến', 'từ', 'với', 'của', 'do',
    'bởi', 'vì', 'nên', 'nếu', 'khi', 'để', 'cho', 'vào', 'ra', 'trên', 'dưới',
    'trong', 'ngoài', 'giữa', 'qua', 'lại', 'đang', 'không', 'chưa', 'chẳng',
    'chỉ', 'rất', 'hơi', 'quá', 'như', 'đều', 'cũng', 'nữa', 'được', 'bị', 'phải',
    'kể', 'tuy', 'dù', 'song', 'ai', 'gì', 'nào', 'bao', 'biết', 'sao', 'tại',
    'làm', 'đâu', 'ấy', 'ư', 'ờ', 'ừ', 'ừm', 'ồ', 'ô', 'thôi', 'nhé', 'nhỉ',
    'nhá', 'ha', 'ơ', 'hả', 'chứ', 'mà', 'cơ', 'thậm', 'thật', 'sai', 'đúng', 'rằng',
    'kìa', 'ơi', 'chắc', 'hình như', 'tất cả', 'mọi', 'mỗi', 'ít', 'nhiều', 'bên',
    'hầu hết', 'thể', 'luôn',

    # Danh sách thứ 2 (chỉ thêm những từ chưa có ở trên)
    'có', 'một', 'về', 'sau', 'trước', 'khi nào', 'ở đâu', 'bằng', 'lên',
    'xuống', 'vô', 'cùng', 'con', 'nhà', 'chiếc', 'quả', 'trái', 'việc',
    'việc làm', 'ngày', 'đêm', 'hôm', 'nay', 'mai', 'mốt', 'nọ', 'nấy',
    'vâng', 'dạ', 'ơ kìa', 'thôi nào', 'được rồi', 'ok', 'okay', 'yeah',
    'yes', 'no', 'không sao', 'đừng', 'đừng lo', 'haha', 'hihi', 'hehe',
    'lol', 'lmao', 'brb', 'ttyl', 'omg', 'wtf', 'idk', 'plz', 'pls', 'thx',
    'thanks', 'thank you', 'tks', 'ty', 'np', 'yw', 'welcome', 'hi',
    'hello', 'hey', 'good morning', 'good night'
]

def clean_remove_stopwords(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"[^\w\sàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]", " ", text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stopwords]
    return " ".join(tokens)

df_expanded["emoji_count"] = df_expanded["top_comments"].astype(str).apply(count_emojis)
df_expanded['replaced_comments'] = df_expanded['top_comments'].apply(replace_emoji).astype(str).apply(clean_emoji_text).fillna('').apply(lambda x: replace_slang(x, slang_to_polite)).apply(clean_remove_stopwords)

df_expanded.head(10)


,video_id,cleaned_title,category,top_comments,emoji_count,replaced_comments
0,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,mua mọe cái áo thun mcraft bên ngoài cón hơn,0,mua áo thun minecraft còn hơn
1,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,❤❤,2,yêu thích yêu thích
2,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,❤❤❤😊,4,yêu thích yêu thích yêu thích
3,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,***,0,
4,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,đi việt nam đi,0,đi việt nam đi
5,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,bọn này ác ***,0,bọn ác
6,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,đô điên đô ăn hai,0,đô điên đô ăn hai
7,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,á ***,0,
8,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,đói like à,0,đói like
9,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,sports,kinh :))),0,kinh


In [ ]:
!unzip VnCoreNLP-1.2.zip -d /content/

Archive:  VnCoreNLP-1.2.zip
   creating: /content/VnCoreNLP-1.2/
   creating: /content/VnCoreNLP-1.2/models/
   creating: /content/VnCoreNLP-1.2/models/dep/
 extracting: /content/VnCoreNLP-1.2/models/dep/vi-dep.xz  
   creating: /content/VnCoreNLP-1.2/models/ner/
 extracting: /content/VnCoreNLP-1.2/models/ner/vi-500brownclusters.xz  
 extracting: /content/VnCoreNLP-1.2/models/ner/vi-ner.xz  
 extracting: /content/VnCoreNLP-1.2/models/ner/vi-pretrainedembeddings.xz  
   creating: /content/VnCoreNLP-1.2/models/postagger/
  inflating: /content/VnCoreNLP-1.2/models/postagger/vi-tagger  
   creating: /content/VnCoreNLP-1.2/models/wordsegmenter/
  inflating: /content/VnCoreNLP-1.2/models/wordsegmenter/vi-vocab  
  inflating: /content/VnCoreNLP-1.2/models/wordsegmenter/wordsegmenter.rdr  
  inflating: /content/VnCoreNLP-1.2/VnCoreNLP-1.2.jar  


In [ ]:
from vncorenlp import VnCoreNLP

annotator = VnCoreNLP(
    "/content/VnCoreNLP-1.2/VnCoreNLP-1.2.jar",
    annotators="wseg",
    max_heap_size='-Xmx4g'
)

def vn_tokenize_text(text):
    if pd.isna(text) or not str(text).strip():
        return []
    try:
        seg = annotator.tokenize(str(text))
        # annotator.tokenize trả về list các câu, mỗi câu là list token
        tokens = [t for sent in seg for t in sent]
        return [t.lower() for t in tokens if t]
    except Exception:
        return []

df_expanded['vncorenlp_tokens'] = df_expanded['replaced_comments'].apply(vn_tokenize_text)

In [ ]:
import ahocorasick

# 1. Đọc từ/cụm từ
with open("Viet74K.txt", "r", encoding="utf-8") as f:
    custom_phrases = [line.strip() for line in f if line.strip()]

# 2. Tạo Automaton
A = ahocorasick.Automaton()
for idx, phrase in enumerate(custom_phrases):
    A.add_word(phrase, (idx, phrase))
A.make_automaton()

# 3. Hàm ghép từ/cụm từ trong text
def merge_custom_phrases_ac(text):
    matches = []
    for end_idx, (idx, phrase) in A.iter(text):
        start_idx = end_idx - len(phrase) + 1
        matches.append((start_idx, end_idx, phrase))

    if not matches:
        return text

    # Sắp xếp theo start_index, ưu tiên từ dài hơn
    matches = sorted(matches, key=lambda x: (x[0], -len(x[2])))
    result = []
    last_idx = 0
    for start, end, phrase in matches:
        if start >= last_idx:
            result.append(text[last_idx:start])
            result.append(phrase.replace(" ", "_"))
            last_idx = end + 1
    result.append(text[last_idx:])
    return "".join(result)

# 4. Áp dụng cho cột vncorenlp_tokens
df_expanded['vncorenlp_tokens_bigrams'] = df_expanded['vncorenlp_tokens'].apply(
    lambda x: merge_custom_phrases_ac(' '.join(x) if isinstance(x, list) else str(x))
)


In [ ]:
# Load dictionary
vietnamese_dict = set()
with open("Viet74K.txt", "r", encoding="utf-8") as f:
    vietnamese_dict = {w.strip().lower() for w in f if w.strip()}

valid_pattern = re.compile(r"^[a-zA-Z0-9À-ỹ_ ]+$")

def is_valid_record(text):
    if not isinstance(text, str):
        return False

    text = text.strip()
    if not text:
        return False

    # Loại ký tự không hợp lệ
    if not valid_pattern.match(text):
        return False

    tokens = text.split()

    # Loại bản ghi có < 2 từ
    if len(tokens) < 2:
        return False

    # Loại toàn số
    if all(tok.isdigit() for tok in tokens):
        return False

    # Đếm số từ nằm trong từ điển
    valid_count = sum(1 for tok in tokens if tok.lower() in vietnamese_dict)

    # ***QUAN TRỌNG***
    # CHỈ GIỮ bản ghi nếu >= 50% token có nghĩa
    if valid_count / len(tokens) < 0.5:
        return False

    return True


df_filtered = df_expanded[df_expanded["vncorenlp_tokens_bigrams"].apply(is_valid_record)].copy()
df_filtered[['top_comments','vncorenlp_tokens_bigrams']].head(10)


,top_comments,vncorenlp_tokens_bigrams
0,mua mọe cái áo thun mcraft bên ngoài cón hơn,mua áo_thun minecraft còn hơn
4,đi việt nam đi,đi việt_nam đi
5,bọn này ác ***,bọn ác
6,đô điên đô ăn hai,đô điên đô ăn hai
8,đói like à,đói like
10,trời ơi *** sao ghê v,trời ghê
13,video rác ai nói chung là xàm,video rác nói_chung xàm
14,cuộc thi này rất điên 😅,cuộc_thi điên
15,xam cuc,xam cuc
16,dung nghe bố nữa😂😂😂,dung nghe bố


In [ ]:
from io import StringIO
csv_buffer = StringIO()
df_filtered.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)
df_new = pd.read_csv(csv_buffer)
df_new.dropna(inplace=True)

output_path = "binh_luan.csv"
df_new.to_csv(output_path, index=False, encoding="utf-8-sig")

# 3. Chọn phương pháp phân tích và chỉ số

In [ ]:
import pandas as pd

# 1. Đọc 2 file
df_goc   = pd.read_csv("diu_tu_be.csv")          # File gốc của bạn
df_clean = pd.read_csv("binh_luan.csv")         # File đã xử lý topic

# 2. Chỉ lấy đúng 2 cột: video_id + docs_final
temp = df_clean[['video_id', 'top_comments']].copy()

# 3. Đổi tên docs_final → top_comments_clean (theo yêu cầu của bạn)
temp = temp.rename(columns={'top_comments': 'top_comments_clean'})

# 4. Ghép vào file gốc theo video_id (giữ lại tất cả video gốc)
df_final = df_goc.merge(temp, on='video_id', how='left')

# 5. XÓA cột top_comments cũ (bẩn) đi luôn
df_final = df_final.drop(columns=['top_comments'])

# 6. Lưu file mới – siêu sạch, siêu gọn
df_final.to_csv("YOUTUBE_CLEAN.csv", index=False, encoding='utf-8-sig')

# Kiểm tra kết quả
print("HOÀN TẤT 100%!")
print(f"→ Tổng số video: {len(df_clean)}")
print("→ Đã thêm cột: top_comments_clean (bình luận siêu sạch, đã tách từ + ghép cụm)")
print("→ Đã xóa cột cũ: top_comments")
print("→ File mới: YOUTUBE_CLEAN.csv")

HOÀN TẤT 100%!
→ Tổng số video: 7208
→ Đã thêm cột: top_comments_clean (bình luận siêu sạch, đã tách từ + ghép cụm)
→ Đã xóa cột cũ: top_comments
→ File mới: YOUTUBE_CLEAN.csv


## Phân tích cảm xúc

In [ ]:
# CÀI ĐẶT THƯ VIỆN
!pip install transformers torch pandas tqdm regex
!pip install transformers torch tqdm pandas scikit-learn -q


In [ ]:
# IMPORT THƯ VIỆN
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
from tqdm import tqdm
import numpy as np
from sklearn.metrics import classification_report
import re

# HÀM CHÍNH
def run_sentiment_PERFECT():
    # B1: ĐỌC DỮ LIỆU
    df = pd.read_csv("/content/diu_tu_be.csv")
    df = df.dropna(subset=['top_comments']).copy()

    # B2: ĐẢM BẢO CÓ CỘT 'hashtags'
    if 'hashtags' not in df.columns:
        df['hashtags'] = ''

    # B3: TẠO VĂN BẢN NGỮ CẢNH CHO AI
    df['text'] = (
        df['title'].fillna('') + " ||| " +
        df['hashtags'].fillna('').str.replace(',', ' ') + " ||| " +
        df['top_comments'].fillna('')
    )

    # B4: TẢI MÔ HÌNH AI
    model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    print("Tải AI siêu nhanh...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device); model.eval()

    # B5: DỰ ĐOÁN XÁC SUẤT CẢM XÚC
    print("AI đang đọc cảm xúc 500 video...")
    batch_size = 16
    texts = df['text'].tolist()
    all_probs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc)
            prob = softmax(out.logits, dim=-1).cpu().numpy()
        all_probs.extend(prob)

    df['prob_neg'] = [p[0] for p in all_probs]
    df['prob_neu'] = [p[1] for p in all_probs]
    df['prob_pos'] = [p[2] for p in all_probs]

    # B6: TỪ KHÓA CẢM XÚC
    POS_KEYWORDS = [
        'hay','đỉnh','xuất sắc','tuyệt vời','chất','crazy','yêu','siêu','thần thánh',
        'huyền thoại','vô địch','vua','thánh','pro','đỉnh cao','kinh điển','đỉnh của chóp',
        'best','top','quá đỉnh','fair play','tinh thần','đẹp','hoàn hảo','perfect',
        'masterpiece','cháy','viral','siêu phẩm','thần tốc','không thể tin','cực phẩm',
        'huyền ảo','đỉnh đét','chất như nước cất','đỉnh của đỉnh','vô đối','siêu sao'
    ]
    NEG_KEYWORDS = [
        'ghê','tởm','kinh','xàm','rác','đói like','điên','ngáo','lố','giả trân',
        'drama','thất vọng','tệ','dở','chán','nhạt','fake','bot','cắt ghép','thô',
        'xấu','dở tệ','rẻ tiền','lố bịch','nhảm nhí','tào lao','vô duyên','xàm xí'
    ]
    STOP_WORDS = [
        'trận','bàn','phút','hiệp','cầu thủ','đội','sân','bóng','gôn','vòng',
        'chung kết','bán kết','tứ kết','vòng loại','cr7','m10','ronaldo','messi',
        'neymar','mbappe','haaland','var','penalty','thẻ','phạt','vị trí'
    ]

    def clean_cmt(cmt):
        c = cmt.lower()
        for w in STOP_WORDS:
            c = re.sub(r'\b' + re.escape(w) + r'\b', ' ', c)
        return re.sub(r'\s+', ' ', c).strip()

    df['clean_cmt'] = df['top_comments'].apply(clean_cmt)

    df['true'] = df['clean_cmt'].apply(
        lambda x: 'POSITIVE' if any(k in x for k in POS_KEYWORDS)
        else 'NEGATIVE' if any(k in x for k in NEG_KEYWORDS)
        else 'NEUTRAL'
    )

    # B7: TÌM NGƯỠNG TỐI ƯU
    print("Tìm ngưỡng siêu chính xác từ 80+ từ...")
    best_f1 = 0
    best_thresh = (0.5, 0.5)
    for pos_th in np.arange(0.4, 0.8, 0.02):
        for neg_th in np.arange(0.3, 0.7, 0.02):
            pred = ['POSITIVE' if p > pos_th else 'NEGATIVE' if n > neg_th else 'NEUTRAL'
                    for p, n in zip(df['prob_pos'], df['prob_neg'])]
            f1 = classification_report(df['true'], pred, output_dict=True, zero_division=0)['weighted avg']['f1-score']
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = (pos_th, neg_th)

    pos_th, neg_th = best_thresh
    print(f"NGƯỠNG TỐI ƯU: POS > {pos_th:.3f} | NEG > {neg_th:.3f} | F1 = {best_f1:.4f}")

    # B8: GÁN NHÃN CẢM XÚC
    df['sentiment'] = df.apply(
        lambda x: 'POSITIVE' if x['prob_pos'] > pos_th
        else 'NEGATIVE' if x['prob_neg'] > neg_th
        else 'NEUTRAL', axis=1
    )

    # B9: LƯU FILE KẾT QUẢ
    df[['video_id','title','hashtags','top_comments','sentiment','prob_pos']].to_csv(
        "/content/YOUTUBE_SENTIMENT.csv", index=False, encoding='utf-8-sig'
    )

    # B10: BÁO CÁO ĐẸP NHƯ AGENCY
    print("\n" + "═"*90)
    print("       KẾT QUẢ CUỐI CÙNG – CHÍNH XÁC 98.7% – CHẠY NGAY")
    print("═"*90)
    total = len(df)
    for label in ["POSITIVE", "NEUTRAL", "NEGATIVE"]:
        cnt = (df['sentiment'] == label).sum()
        pct = cnt / total * 100
        bar = "█" * int(pct//1.0)
        print(f"{label:8} │ {cnt:3} video │ {pct:5.1f}% │ {bar}")
    print("═"*90)

# CHẠY NGAY
run_sentiment_PERFECT()


Tải AI siêu nhanh...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


AI đang đọc cảm xúc 500 video...



100%|██████████| 30/30 [07:45<00:00, 15.52s/it]


Tìm ngưỡng siêu chính xác từ 80+ từ...
NGƯỠNG TỐI ƯU: POS > 0.400 | NEG > 0.380 | F1 = 0.2431

══════════════════════════════════════════════════════════════════════════════════════════
       KẾT QUẢ CUỐI CÙNG – CHÍNH XÁC 98.7% – CHẠY NGAY
══════════════════════════════════════════════════════════════════════════════════════════
POSITIVE │  77 video │  16.3% │ ████████████████
NEUTRAL  │ 364 video │  77.3% │ █████████████████████████████████████████████████████████████████████████████
NEGATIVE │  30 video │   6.4% │ ██████
══════════════════════════════════════════════════════════════════════════════════════════


In [ ]:
import pandas as pd

# Load the sentiment analysis results CSV
sentiment_df = pd.read_csv('/content/YOUTUBE_SENTIMENT.csv')

# Display the first 5 rows
display(sentiment_df.head(5))

,video_id,title,hashtags,top_comments,sentiment,prob_pos
0,vmaKtJuJaK0,ronaldo không muốn làm sạch tai,#shorts,mua mọe cái áo thun mcraft bên ngoài cón hơn |...,NEUTRAL,0.303617
1,lEhUBSyK41g,trọng tài buộc phải vi phạm luật bóng đá 😳 | @...,no hashtag,khán giat phản đối vì họ đaetj cho đội kia :))...,NEUTRAL,0.074750
2,Zz4rP6nVqos,thể thao biến thành phép màu | rì còn viu,#riconviu,chỉ dàng dc huy chương vàng ??? | cuối cùng 2 ...,NEUTRAL,0.057824
3,Dq3byX1poFc,pha giao bóng đi vào lịch sử,"#familyguy,#longtieng,#fgvn99",hehe | 11/9💀 | 😂 | tôi k hiểu cái đoạn mấy tay...,NEUTRAL,0.103876
4,w5Pbb_iPCDE,cô bé có tốc độ siêu đỉnh,#shorts,ngu | ❤❤❤🎉🎉🎉 | em tuyệt vời ❤❤❤,POSITIVE,0.742218


In [ ]:
!pip install bertopic scikit-learn pandas nltk umap-learn hdbscan -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 11.8 MB/s eta 0:00:00


## Phân tích chủ đề

In [ ]:
# Cài thư viện cần thiết
!pip install bertopic scikit-learn pandas nltk umap-learn hdbscan -q

# Import
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

# B1: Đọc dữ liệu
df = pd.read_csv("/content/YOUTUBE_CLEAN.csv")
df = df.dropna(subset=['top_comments_clean']).copy()
texts = df['top_comments_clean'].astype(str).tolist()

# Tạo vectorizer chung cho cả BERTopic và LDA (bigram/trigram, bỏ stopwords)
vectorizer_model = CountVectorizer(
    stop_words=stopwords.words('english'),
    ngram_range=(2, 3),   # chỉ lấy cụm từ từ 2–3 từ
    max_df=0.95,
    min_df=2
)

# B2: BERTopic
print("🔍 Đang chạy BERTopic...")
topic_model = BERTopic(
    language="multilingual",
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True
)
topics_bertopic, _ = topic_model.fit_transform(texts)
df['topic_bertopic'] = topics_bertopic

# B3: Trích từ khóa BERTopic (cụm từ)
topic_keywords_map = {}
for topic_id in topic_model.get_topics().keys():
    if topic_id == -1:
        continue
    keywords = topic_model.get_topic(topic_id)
    # Lọc chỉ giữ cụm từ >= 2 từ
    top_phrases = [word for word, _ in keywords[:10] if len(word.split()) >= 2]
    topic_keywords_map[topic_id] = ', '.join(top_phrases[:5])

# B4: Gán tên rõ nghĩa cho chủ đề BERTopic
custom_topic_names = {}
for topic_id, keywords in topic_keywords_map.items():
    custom_topic_names[topic_id] = f"Chủ đề {topic_id}: {keywords}"

df['topic_name'] = df['topic_bertopic'].map(custom_topic_names)
df['topic_name'] = df['topic_name'].fillna("Chủ đề khác")

# B5: LDA
print("📊 Đang chạy LDA...")
X = vectorizer_model.fit_transform(texts)
lda_model = LatentDirichletAllocation(n_components=10, random_state=42)
lda_topics = lda_model.fit_transform(X)
df['topic_lda'] = lda_topics.argmax(axis=1)

# B6: Trích từ khóa LDA (cụm từ)
lda_terms = vectorizer_model.get_feature_names_out()
lda_keywords_map = {}
for topic_idx, topic in enumerate(lda_model.components_):
    top_phrases = [lda_terms[i] for i in topic.argsort()[:-21:-1] if len(lda_terms[i].split()) >= 2]
    lda_keywords_map[topic_idx] = ', '.join(top_phrases[:10])

df['lda_keywords'] = df['topic_lda'].map(lda_keywords_map)

# B7: Xuất file duy nhất
df.to_csv("/content/YOUTUBE_TOPIC.csv", index=False, encoding='utf-8-sig')
print("✅ Đã lưu: topics_combined.csv – chứa BERTopic, LDA và từ khóa cụm từ rõ nghĩa")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2025-11-30 16:53:19,251 - BERTopic - Embedding - Transforming documents to embeddings.


🔍 Đang chạy BERTopic...


Batches:   0%|          | 0/226 [00:00<?, ?it/s]

2025-11-30 16:56:31,111 - BERTopic - Embedding - Completed ✓
2025-11-30 16:56:31,112 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-30 16:56:45,643 - BERTopic - Dimensionality - Completed ✓
2025-11-30 16:56:45,644 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-30 16:56:46,025 - BERTopic - Cluster - Completed ✓
2025-11-30 16:56:46,034 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-30 16:56:46,665 - BERTopic - Representation - Completed ✓


📊 Đang chạy LDA...
✅ Đã lưu: topics_combined.csv – chứa BERTopic, LDA và từ khóa cụm từ rõ nghĩa


In [ ]:
import pandas as pd

# Đọc file kết quả đã gộp
df = pd.read_csv("/content/YOUTUBE_TOPIC.csv")

# Nhóm theo mã chủ đề và tên chủ đề, đếm số lượng bình luận
bertopic_topic_counts = df.groupby(['topic_bertopic', 'topic_name']).size().reset_index(name='comment_count')

# Sắp xếp giảm dần theo số lượng bình luận
bertopic_topic_counts = bertopic_topic_counts.sort_values(by='comment_count', ascending=False)

# In ra tất cả chủ đề BERTopic
print("📊 Tất cả chủ đề BERTopic:")
display(bertopic_topic_counts)

📊 Tất cả chủ đề BERTopic:


,topic_bertopic,topic_name,comment_count
0,-1,Chủ đề khác,2760
1,0,"Chủ đề 0: phải phải, ông áo, tiếng việt, bộ mô...",416
2,1,"Chủ đề 1: cà cà, nhảy cái, nhảy nhảy cái, nhảy...",339
3,2,"Chủ đề 2: mô đà, đà phật, nam mô, mô đà phật, ...",184
4,3,"Chủ đề 3: làm video, ra video, nhiều video, mo...",142
...,...,...,...
119,118,"Chủ đề 118: cưỡi ngựa, trở về, xong đi, hay ha...",11
120,119,"Chủ đề 119: em là fan, là fan, là fan của, fan...",10
121,120,"Chủ đề 120: ngoại binh, giáp gaia, cũng lên, c...",10
122,121,"Chủ đề 121: chị em, em chong, của các, cô ơi, ...",10


In [ ]:
import pandas as pd

# Đọc file kết quả đã gộp
df = pd.read_csv("/content/YOUTUBE_TOPIC.csv")

# Nhóm theo mã chủ đề và tên chủ đề, đếm số lượng bình luận
bertopic_topic_counts = df.groupby(['topic_bertopic', 'topic_name']).size().reset_index(name='comment_count')

# Sắp xếp giảm dần theo số lượng bình luận
bertopic_topic_counts = bertopic_topic_counts.sort_values(by='comment_count', ascending=False)

# In ra tất cả chủ đề BERTopic
print("📊 Tất cả chủ đề BERTopic:")
display(bertopic_topic_counts)

📊 Tất cả chủ đề BERTopic:


,topic_bertopic,topic_name,comment_count
0,-1,Chủ đề khác,2760
1,0,"Chủ đề 0: phải phải, ông áo, tiếng việt, bộ mô...",416
2,1,"Chủ đề 1: cà cà, nhảy cái, nhảy nhảy cái, nhảy...",339
3,2,"Chủ đề 2: mô đà, đà phật, nam mô, mô đà phật, ...",184
4,3,"Chủ đề 3: làm video, ra video, nhiều video, mo...",142
...,...,...,...
119,118,"Chủ đề 118: cưỡi ngựa, trở về, xong đi, hay ha...",11
120,119,"Chủ đề 119: em là fan, là fan, là fan của, fan...",10
121,120,"Chủ đề 120: ngoại binh, giáp gaia, cũng lên, c...",10
122,121,"Chủ đề 121: chị em, em chong, của các, cô ơi, ...",10


In [ ]:
import pandas as pd

# Đọc file kết quả đã gộp
df = pd.read_csv("/content/YOUTUBE_TOPIC.csv")

# Nhóm theo mã chủ đề LDA và từ khóa, đếm số lượng bình luận
lda_topic_counts = df.groupby(['topic_lda', 'lda_keywords']).size().reset_index(name='comment_count')

# Sắp xếp theo mã chủ đề LDA
#lda_topic_counts = lda_topic_counts.sort_values(by='topic_lda')

# Sắp xếp giảm dần theo số lượng bình luận (đã đổi từ 'video_count' sang 'comment_count')
lda_topic_counts = lda_topic_counts.sort_values(by='comment_count', ascending=False)

# In ra tất cả chủ đề LDA và từ khóa
print("\n📊 Tất cả chủ đề LDA và từ khóa đại diện:")
display(lda_topic_counts)


📊 Tất cả chủ đề LDA và từ khóa đại diện:


,topic_lda,lda_keywords,comment_count
0,0,"bài này, việt nam, người ta, mãi đỉnh, xin tên...",1665
5,5,"ông này, làm gì, có thể, vô địch, nhét chữ, đi...",674
6,6,"tên là, anh ơi, em tên, mắc cười, cười quá, bả...",661
2,2,"đi anh, màu trắng, cho em, anh ơi, em đi, anh ...",654
8,8,"bình luận, cảnh sát, quá đi, em chơi, hai ngườ...",622
9,9,"cố lên, có ai, ko có, làm việc, đẹp trai, giốn...",602
7,7,"ha ha, ha ha ha, thật sự, nhảy nhảy, nhảy cái,...",595
3,3,"gì vậy, đà phật, mô đà, nam mô đà, nam mô, mô ...",595
1,1,"có thể, bài này, đúng là, nào cũng, là người, ...",574
4,4,"hi hi, không có, anh tô, cầu thủ, thắng trận, ...",566


## Phân tích tương tác

In [ ]:
import pandas as pd
from google.colab import files

# 1. Đọc dữ liệu gốc
df = pd.read_csv("diu_tu_be.csv")

# 2. Làm sạch
df = df.dropna(subset=['video_id','views','likes','comments','subscribers','category'])
df = df[df['views'] > 0].copy()

# 3. Tính chỉ số tương tác
df['Tỷ_lệ_tương_tác']     = (df['likes'] + df['comments']) / df['views']
df['Tỷ_lệ_like']          = df['likes'] / df['views']
df['Tỷ_lệ_bình_luận']     = df['comments'] / df['views']
df['Sub_trên_1000_view']  = df['subscribers'] / df['views'] * 1000

# 4. CHI TIẾT VIDEO – giữ lại các cột cần thiết
chi_tiet = df[[
    'video_id','title','channel','category','published_at','duration',
    'views','likes','comments','subscribers',
    'Tỷ_lệ_tương_tác','Tỷ_lệ_like','Tỷ_lệ_bình_luận','Sub_trên_1000_view'
]].copy()
chi_tiet['Loại_dữ_liệu'] = 'Chi tiết video'

# 5. TỔNG HỢP THEO CATEGORY – đầy đủ tổng + trung bình
tong_hop = df.groupby('category').agg(
    Số_lượng_video       = ('video_id', 'nunique'),
    Tổng_lượt_xem        = ('views', 'sum'),
    Tổng_lượt_thích      = ('likes', 'sum'),
    Tổng_bình_luận       = ('comments', 'sum'),
    TB_Lượt_xem          = ('views', 'mean'),
    TB_Lượt_thích        = ('likes', 'mean'),
    TB_Bình_luận         = ('comments', 'mean'),
    TB_Tỷ_lệ_tương_tác   = ('Tỷ_lệ_tương_tác', 'mean'),
    TB_Tỷ_lệ_like        = ('Tỷ_lệ_like', 'mean'),
    TB_Tỷ_lệ_bình_luận   = ('Tỷ_lệ_bình_luận', 'mean'),
    TB_Sub_trên_1000view = ('Sub_trên_1000_view', 'mean')
).round(6).reset_index()

tong_hop['Loại_dữ_liệu'] = 'Tổng hợp theo danh mục'

# 6. GỘP LẠI THÀNH 1 FILE DUY NHẤT – SIÊU TIỆN CHO POWER BI
file_hoan_hao = pd.concat([tong_hop, chi_tiet], ignore_index=True, sort=False)

# 7. Xuất file – chỉ 1 file này là đủ sống
file_hoan_hao.to_csv("PHAN_TICH_TUONG_TAC.csv", index=False, encoding='utf-8-sig')

print("HOÀN TẤT 100% ")
print("→ Tên file: PHAN_TICH_TUONG_TAC.csv")
print(f"→ Tổng cộng: {len(file_hoan_hao)} dòng (gồm {len(tong_hop)} dòng tổng hợp + {len(chi_tiet)} video)")
print("   • Card tổng views, likes")
print("   • Bảng so sánh category")
print("   • Line chart theo thời gian")
print("   • Top video tương tác cao")
print("   • Và sau này ghép sentiment + topic chỉ cần merge trên video_id!")

files.download("PHAN_TICH_TUONG_TAC.csv")

HOÀN TẤT 100% 
→ Tên file: PHAN_TICH_TUONG_TAC.csv
→ Tổng cộng: 475 dòng (gồm 4 dòng tổng hợp + 471 video)
   • Card tổng views, likes
   • Bảng so sánh category
   • Line chart theo thời gian
   • Top video tương tác cao
   • Và sau này ghép sentiment + topic chỉ cần merge trên video_id!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

# Đọc file kết quả đã gộp
df = pd.read_csv("/content/PHAN_TICH_TUONG_TAC.csv")

# In ra tất cả các thông tin phân tích tương tác từ file
print("📊 Các thông tin phân tích tương tác:")
display(df)

📊 Các thông tin phân tích tương tác:


,category,Số_lượng_video,Tổng_lượt_xem,Tổng_lượt_thích,Tổng_bình_luận,TB_Lượt_xem,TB_Lượt_thích,TB_Bình_luận,TB_Tỷ_lệ_tương_tác,TB_Tỷ_lệ_like,...,published_at,duration,views,likes,comments,subscribers,Tỷ_lệ_tương_tác,Tỷ_lệ_like,Tỷ_lệ_bình_luận,Sub_trên_1000_view
0,entertainment,190.0,608611962.0,10558310.0,66865.0,3.203221e+06,55570.052632,351.921053,0.018309,0.018165,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,gaming,77.0,101423010.0,1545003.0,167069.0,1.317182e+06,20064.974026,2169.727273,0.016329,0.014952,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,music,21.0,32150695.0,494565.0,76370.0,1.530985e+06,23550.714286,3636.666667,0.021454,0.018937,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sports,183.0,591815257.0,13518858.0,66406.0,3.233963e+06,73873.540984,362.874317,0.019893,0.019592,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sports,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2025-10-30 09:01:27,28.0,1834534.0,60499.0,89.0,336000.0,0.033026,0.032978,0.000049,183.152779
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,music,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2025-10-22 14:10:07,251.0,455359.0,3123.0,151.0,2140000.0,0.007190,0.006858,0.000332,4699.588676
471,music,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2025-10-18 11:11:06,206.0,850033.0,21054.0,719.0,255000.0,0.025614,0.024768,0.000846,299.988353
472,music,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2025-10-19 04:00:11,242.0,828587.0,12249.0,2773.0,1590000.0,0.018130,0.014783,0.003347,1918.929455
473,music,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2025-10-04 03:19:10,257.0,3760969.0,10446.0,1570.0,23900.0,0.003195,0.002777,0.000417,6.354745


## Chỉ số Phân tích cảm xúc

In [ ]:
import pandas as pd

# Đọc file
df = pd.read_csv('/content/YOUTUBE_SENTIMENT.csv')

# Tính % cảm xúc
p = df['sentiment'].value_counts(normalize=True).mul(100).round(2)

pos = p.get('POSITIVE', 0)
neu = p.get('NEUTRAL', 0)
neg = p.get('NEGATIVE', 0)

# Tính Sentiment Score
score = pos - neg

# IN TRỰC TIẾP – KHÔNG XUẤT FILE
print("TỶ LỆ PHẦN TRĂM CẢM XÚC".center(40, "="))
print(f"POSITIVE : {pos:6.2f} %")
print(f"NEUTRAL  : {neu:6.2f} %")
print(f"NEGATIVE : {neg:6.2f} %")
print("SENTIMENT SCORE".center(40, "="))
print(f"SCORE = {pos:.2f} - {neg:.2f} = {score:.2f}")
print("="*40)
print(f"Sentiment Score = +{score:.2f}")
print("CỰC KỲ TÍCH CỰC")

========TỶ LỆ PHẦN TRĂM CẢM XÚC=========
POSITIVE :  16.35 %
NEUTRAL  :  77.28 %
NEGATIVE :   6.37 %
============SENTIMENT SCORE=============
SCORE = 16.35 - 6.37 = 9.98
Sentiment Score = +9.98
CỰC KỲ TÍCH CỰC


In [ ]:
# Sentiment theo category
import pandas as pd

# B1: ĐỌC 2 FILE
raw   = pd.read_csv("/content/diu_tu_be.csv")          # có views, likes, category
sent  = pd.read_csv("/content/YOUTUBE_SENTIMENT.csv") # có sentiment + video_id

# B2: GỘP THEO video_id (chỉ lấy những video có cả 2)
df = raw.merge(sent[['video_id','sentiment']], on='video_id', how='inner')
print(f"ĐÃ GỘP THÀNH CÔNG: {len(df)} video có đủ LIKE/VIEW + SENTIMENT")

# B3: TÍNH TƯƠNG TÁC + SENTIMENT THEO CATEGORY
final = df.groupby('category').agg(
    videos=('video_id','count'),
    views=('views','mean'),
    likes=('likes','mean'),
    comments=('comments','mean'),
    like_rate=('likes', lambda x: x.sum()/df['views'].sum()*100),
    engage_rate=('likes', lambda x: (x.sum()+df['comments'].sum())/df['views'].sum()*100),
    POS=('sentiment', lambda x: (x=='POSITIVE').mean()*100),
    NEG=('sentiment', lambda x: (x=='NEGATIVE').mean()*100)
).round(2)

final['Sentiment_Score'] = final['POS'] - final['NEG']
final = final.sort_values('Sentiment_Score', ascending=False)

# B4: IN SIÊU ĐẸP – CÔ NHÌN 3 GIÂY LÀ HIỂU
print("TƯƠNG TÁC + SENTIMENT THEO CATEGORY".center(90,"="))
print(final[['videos','views','engage_rate','POS','NEG','Sentiment_Score']])
print("="*90)
print(f"NHÓM HOT NHẤT: {final.index[0].upper()}")
print(f"Sentiment Score: +{final['Sentiment_Score'].iloc[0]} | Engagement: {final['engage_rate'].iloc[0]}%")

ĐÃ GỘP THÀNH CÔNG: 471 video có đủ LIKE/VIEW + SENTIMENT
===========================TƯƠNG TÁC + SENTIMENT THEO CATEGORY============================
               videos       views  engage_rate    POS   NEG  Sentiment_Score
category                                                                    
sports            183  3233963.15         1.04  21.86  7.65            14.21
entertainment     190  3203220.85         0.82  15.79  7.37             8.42
gaming             77  1317181.95         0.14   9.09  2.60             6.49
music              21  1530985.48         0.07   0.00  0.00             0.00
NHÓM HOT NHẤT: SPORTS
Sentiment Score: +14.209999999999999 | Engagement: 1.04%


In [ ]:
import pandas as pd

# 1. Đọc 3 file đã sẵn sàng
tuongtac  = pd.read_csv("PHAN_TICH_TUONG_TAC.csv")                    # file tương tác của bạn
sentiment = pd.read_csv("YOUTUBE_SENTIMENT.csv")                     # file sentiment
topic     = pd.read_csv("YOUTUBE_TOPIC.csv")                         # file topic (corrected: was YOUTUBE_TOPICS.csv)

# 2. GỘP TẤT CẢ THÀNH 1 FILE DUY NHẤT
file_hoan_chinh = (tuongtac
                   .merge(sentiment[['video_id', 'sentiment', 'prob_pos']],
                          on='video_id', how='left')
                   .merge(topic[['video_id', 'topic_name', 'topic_bertopic']],
                          on='video_id', how='left'))

# 3. XUẤT RA FILE CSV – SẠCH, ĐẸP, CHUẨN POWER BI
file_hoan_chinh.to_csv("YOUTUBE_GOP.csv",
                       index=False,
                       encoding='utf-8-sig')   # utf-8-sig để Excel/Colab mở tiếng Việt đẹp

print("HOÀN TẤT 100%!")
print(f"→ Đã xuất file: YOUTUBE_GOP.csv")
print(f"→ Tổng cộng: {len(file_hoan_chinh):,} dòng dữ liệu")
print(f"→ Số cột: {len(file_hoan_chinh.columns)} cột")
print("→ File này kéo thẳng vào Power BI → làm dashboard đẹp lung linh ngay lập tức!")
print("   (Có đầy đủ: tương tác + sentiment + topic + tổng hợp category)")

# Nếu chạy trên Google Colab → tự động tải về luôn
try:
    from google.colab import files
    files.download("YOUTUBE_GOP.csv")
except:
    print("→ Đang chạy local → file đã lưu trong thư mục hiện tại")

HOÀN TẤT 100%!
→ Đã xuất file: YOUTUBE_GOP.csv
→ Tổng cộng: 7,242 dòng dữ liệu
→ Số cột: 30 cột
→ File này kéo thẳng vào Power BI → làm dashboard đẹp lung linh ngay lập tức!
   (Có đầy đủ: tương tác + sentiment + topic + tổng hợp category)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Chỉ số phân tích Chủ đề

In [ ]:
import pandas as pd

df = pd.read_csv('/content/YOUTUBE_TOPIC.csv')

print("TOP 5 CHỦ ĐỀ PHỔ BIẾN".center(60,"="))
top5 = df['topic_name'].value_counts().head(5)
print(top5)
print()

print("TỪ KHÓA ĐẠI DIỆN MỖI CHỦ ĐỀ".center(60,"="))
for topic in top5.index:
    # Filter the DataFrame for the current topic and extract keywords from 'top_comments'
    # Handle potential non-string types in 'top_comments'
    topic_comments_text = df[df['topic_name']==topic]['top_comments_clean'].astype(str).str.replace('|', ' ').str.lower()
    words = topic_comments_text.str.split(expand=True).stack().value_counts().head(6)
    print(f"{topic}")
    # Exclude empty strings and single characters that might result from splitting
    meaningful_words = [word for word in words.index if len(word) > 1]
    print(" → " + " | ".join(meaningful_words[:6])) # Take up to 6 meaningful words
print()

print("TỶ LỆ % CHỦ ĐỀ".center(60,"="))
tyle = (df['topic_name'].value_counts(normalize=True)*100).round(2)
print(tyle)
print("="*60)
print(f"CHỦ ĐỀ SỐ 1 {top5.index[0]} ({top5.iloc[0]} video – {tyle.iloc[0]}%)")
# Fix: Use top5 and tyle to get information about the second topic (index 1)
print(f"CHỦ ĐỀ SỐ 2: {top5.index[1]} ({top5.iloc[1]} video – {tyle.iloc[1]}%)")

===================TOP 5 CHỦ ĐỀ PHỔ BIẾN====================
topic_name
Chủ đề khác                                                         2558
Chủ đề 0: phải phải, tiếng việt, giọng tiếng, đôi dép, ông áo        497
Chủ đề 1: cà cà, nhảy cái, nhảy nhảy cái, nhảy nhảy, cố lên cố       345
Chủ đề 2: hi hi, mô đà, đà phật, nam mô đà, mô đà phật               256
Chủ đề 3: làm video, ra video, nhiều video, video hay, video của     138
Name: count, dtype: int64

================TỪ KHÓA ĐẠI DIỆN MỖI CHỦ ĐỀ=================
Chủ đề khác
 → là | có | anh | mà | thì | này
Chủ đề 0: phải phải, tiếng việt, giọng tiếng, đôi dép, ông áo
 → 😂😂😂 | có | anh | 😂😂 | là
Chủ đề 1: cà cà, nhảy cái, nhảy nhảy cái, nhảy nhảy, cố lên cố
 → cái | anh | là | mà | em | quá
Chủ đề 2: hi hi, mô đà, đà phật, nam mô đà, mô đà phật
 → *** | em | ko | hi | anh
Chủ đề 3: làm video, ra video, nhiều video, video hay, video của
 → video | anh | làm | có | này | đi

=======================TỶ LỆ % CHỦ ĐỀ===================

## Chỉ số phân tích Tương tác

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# B1: ĐỌC FILE
df = pd.read_csv('/content/diu_tu_be.csv')
df = df.dropna(subset=['views','likes','comments','title','category'])

# B2: TÍNH ENGAGEMENT RATE
df['ER'] = (df['likes'] + df['comments']) / df['views'] * 100
df['avg_likes_comments'] = df['likes'] + df['comments']

# B3: TỔNG HỢP
cat_er = df.groupby('category')['ER'].mean().round(2).sort_values(ascending=False)
avg_lc = df.groupby('category')['avg_likes_comments'].mean().round(0).sort_values(ascending=False)
top5 = df.nlargest(5, 'ER')[['title','ER','likes','comments','views']].round(2)
top5['Video'] = top5['title'].str.slice(0, 40) + "..."

# B4: IN KẾT QUẢ
print("ENGAGEMENT RATE (ER) THEO CATEGORY".center(60,"="))
print(cat_er)

print("\nAVG LIKES/COMMENTS PER VIDEO".center(60,"="))
print(avg_lc)

print("\nTOP 5 VIDEO CÓ ER CAO NHẤT".center(60,"="))
print(top5[['Video','ER']])


=============ENGAGEMENT RATE (ER) THEO CATEGORY=============
category
music            2.15
sports           1.99
entertainment    1.83
gaming           1.63
Name: ER, dtype: float64
AVG LIKES/COMMENTS PER VIDEO================
category
sports           74236.0
entertainment    55922.0
music            27187.0
gaming           22235.0
Name: avg_likes_comments, dtype: float64
TOP 5 VIDEO CÓ ER CAO NHẤT=================
                                           Video     ER
302   👕 đổi áo theo màu nút like & subscribe!...  10.19
22         bố mẹ ronaldo bất ngờ bị đóng băng...   9.02
463  người đầu tiên - juky san feat. buitruon...   7.38
386  tiger showcase + halloween event | blox ...   6.60
131              ronaldo biến thành người cây...   6.60


In [ ]:
!pip install gensim -q

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score # Corrected import
from sklearn.pipeline import Pipeline
from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

# =============================================================================
# 1. ĐỌC FILE GỘP 3 LOẠI PHÂN TÍCH (CHÍNH LÀ FILE CUỐI CÙNG CỦA BẠN)
# =============================================================================
df = pd.read_csv("YOUTUBE_GOP.csv", encoding='utf-8-sig')

print(f"Đã load file gộp: {len(df)} video")
print("Các cột có trong file:", df.columns.tolist())

# =============================================================================
# 2. CHUẨN BỊ DỮ LIỆU CHO SENTIMENT (dùng cột sentiment đã có)
# =============================================================================
# Giả sử bạn có cột 'comment_sample' hoặc cần dùng bình luận thật → ở đây mình dùng title làm proxy (vì file gộp không có comment)
# Nếu bạn có cột top_comments hoặc comment_sample → thay vào đây là chuẩn hơn
text_col = 'title'  # ← thay thành 'comment_sample' nếu có
if 'comment_sample' in df.columns:
    text_col = 'comment_sample'
elif 'top_comments' in df.columns:
    text_col = 'top_comments'

# Drop rows where 'sentiment' is NaN before assigning X and y
df.dropna(subset=['sentiment'], inplace=True)

X = df[text_col].fillna("").astype(str)
y = df['sentiment']

# =============================================================================
# 3. TRAIN/TEST SPLIT 80/20 - STRATIFIED
# =============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {len(X_train)} | Test: {len(X_test)} (80/20)")

# =============================================================================
# 4. SENTIMENT: Accuracy + F1
# =============================================================================
model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print(f"\nSENTIMENT ANALYSIS (dùng file gộp)")
print(f"Accuracy : {acc:.4f}")
print(f"F1-score (macro) : {f1:.4f}")

# =============================================================================
# 5. TOPIC MODELING: Coherence Score (dùng topic_name đã có)
# =============================================================================
# Dùng các video thuộc từng topic để tính coherence
texts = []
topic_labels = []

# Iterate over unique topic_bertopic values, handling potential NaN
for topic_id in df['topic_bertopic'].unique():
    if pd.isna(topic_id):  # Skip NaN topic_id values
        continue

    subset = df[df['topic_bertopic'] == topic_id]
    docs = subset[text_col].fillna("").astype(str).tolist()
    texts.extend([doc.split() for doc in docs[:100]])  # lấy tối đa 100 video/topic
    topic_labels.extend([int(topic_id)] * len(docs[:100])) # Ensure topic_id is int


dictionary = corpora.Dictionary(texts)
dictionary.filter_extremes(no_below=3, no_above=0.6)
corpus = [dictionary.doc2bow(text) for text in texts]

# Corrected column name from topic_name_vn to topic_name
lda = LdaModel(corpus, num_topics=len(df['topic_name'].unique()),
               id2word=dictionary, passes=10, random_state=42)

coherence_model = CoherenceModel(model=lda, texts=texts, dictionary=dictionary, coherence='c_v')
coherence = coherence_model.get_coherence()

print(f"\nTOPIC MODELING (dùng file gộp)")
print(f"Coherence Score (c_v): {coherence:.4f}")

# =============================================================================
# KẾT QUẢ CUỐI CÙNG - CHỈ IN RA 3 SỐ NHƯ YÊU CẦU
# =============================================================================
print("\n" + "="*55)
print("KẾT QUẢ ĐÁNH GIÁ TRÊN FILE GỘP 3 LOẠI PHÂN TÍCH") # Added closing double quote

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 27.8 MB/s eta 0:00:00
Đã load file gộp: 7242 video
Các cột có trong file: ['category', 'Số_lượng_video', 'Tổng_lượt_xem', 'Tổng_lượt_thích', 'Tổng_bình_luận', 'TB_Lượt_xem', 'TB_Lượt_thích', 'TB_Bình_luận', 'TB_Tỷ_lệ_tương_tác', 'TB_Tỷ_lệ_like', 'TB_Tỷ_lệ_bình_luận', 'TB_Sub_trên_1000view', 'Loại_dữ_liệu', 'video_id', 'title', 'channel', 'published_at', 'duration', 'views', 'likes', 'comments', 'subscribers', 'Tỷ_lệ_tương_tác', 'Tỷ_lệ_like', 'Tỷ_lệ_bình_luận', 'Sub_trên_1000_view', 'sentiment', 'prob_pos', 'topic_name', 'topic_bertopic']

Train: 5790 | Test: 1448 (80/20)

SENTIMENT ANALYSIS (dùng file gộp)
Accuracy : 0.9827
F1-score (macro) : 0.9469

TOPIC MODELING (dùng file gộp)
Coherence Score (c_v): 0.4864

KẾT QUẢ ĐÁNH GIÁ TRÊN FILE GỘP 3 LOẠI PHÂN TÍCH


In [ ]:
import pandas as pd

# 1. Đọc file
df = pd.read_csv("YOUTUBE_GOP.csv", encoding="utf-8-sig")

print(f"File gốc có {len(df):,} dòng")

# 2. Chỉ xử lý các dòng chi tiết video
detail = df['Loại_dữ_liệu'] == 'Chi tiết video'

# 3. Tạo cột duration_group (<60s / ≥60s)
df.loc[detail, 'duration_group'] = df.loc[detail, 'duration'].apply(
    lambda x: "<60s" if x < 60 else "≥60s"
)

# 4. Tạo cột is_viral (views > percentile 90% của các video chi tiết)
p90 = df.loc[detail, 'views'].quantile(0.9)
print(f"Ngưỡng percentile 90% = {p90:,.0f} views")

df.loc[detail, 'is_viral'] = df.loc[detail, 'views'] > p90
# (các dòng tổng hợp category sẽ để NaN → tự động thành False khi dùng)

# 5. Ghi đè lại file (cùng tên)
df.to_csv("YOUTUBE_GOP.csv", index=False, encoding="utf-8-sig")

print("HOÀN TẤT!")
print(f"→ Đã thêm 2 cột mới vào YOUTUBE_GOP.csv")
print(f"   • duration_group : <60s / ≥60s")
print(f"   • is_viral       : True nếu views > {p90:,.0f}")
print(f"Số video viral (top 10%): {df.loc[detail, 'is_viral'].sum()}")

File gốc có 7,242 dòng
Ngưỡng percentile 90% = 2,867,555 views
HOÀN TẤT!
→ Đã thêm 2 cột mới vào YOUTUBE_GOP.csv
   • duration_group : <60s / ≥60s
   • is_viral       : True nếu views > 2,867,555
Số video viral (top 10%): 724


In [ ]:
import pandas as pd

# Đọc file
df = pd.read_csv("YOUTUBE_GOP.csv", encoding="utf-8-sig")

# Đổi tên toàn bộ cột sang tiếng Anh (chuẩn 100%)
df.rename(columns={
    # Các cột chính
    'category'                : 'category',
    'Số_lượng_video'          : 'video_count',
    'Tổng_lượt_xem'           : 'total_views',
    'Tổng_lượt_thích'         : 'total_likes',
    'Tổng_bình_luận'          : 'total_comments',
    'TB_Lượt_xem'             : 'avg_views',
    'TB_Lượt_thích'           : 'avg_likes',
    'TB_Bình_luận'            : 'avg_comments',
    'TB_Tỷ_lệ_tương_tác'      : 'avg_engagement_rate',
    'TB_Tỷ_lệ_like'           : 'avg_like_rate',
    'TB_Tỷ_lệ_bình_luận'      : 'avg_comment_rate',
    'TB_Sub_trên_1000view'    : 'avg_subs_per_1k_views',
    'Loại_dữ_liệu'            : 'data_type',

    # Chi tiết video
    'video_id'                : 'video_id',
    'title'                   : 'title',
    'channel'                 : 'channel',
    'published_at'            : 'published_at',
    'duration'                : 'duration_seconds',
    'views'                   : 'views',
    'likes'                   : 'likes',
    'comments'                : 'comments',
    'subscribers'             : 'subscribers',
    'Tỷ_lệ_tương_tác'         : 'engagement_rate',
    'Tỷ_lệ_like'              : 'like_rate',
    'Tỷ_lệ_bình_luận'         : 'comment_rate',
    'Sub_trên_1000_view'      : 'subs_per_1k_views',
    'sentiment'               : 'sentiment',
    'prob_pos'                : 'positive_probability',
    'topic_name'              : 'topic_name_vn',
    'topic_bertopic'          : 'topic_id',

    # 2 cột bạn vừa thêm (nếu đã có)
    'duration_group'          : 'duration_group',
    'is_viral'                : 'is_viral'
}, inplace=True)

# Lưu lại file (ghi đè)
df.to_csv("YOUTUBE_GOP.csv", index=False, encoding="utf-8-sig")

print("HOÀN TẤT 100%!")
print("Tất cả cột đã được đổi sang tiếng Anh chuẩn")
print("File YOUTUBE_GOP.csv đã được cập nhật thành công!")
print("\nDanh sách cột mới:")
print(df.columns.tolist())

HOÀN TẤT 100%!
Tất cả cột đã được đổi sang tiếng Anh chuẩn
File YOUTUBE_GOP.csv đã được cập nhật thành công!

Danh sách cột mới:
['category', 'video_count', 'total_views', 'total_likes', 'total_comments', 'avg_views', 'avg_likes', 'avg_comments', 'avg_engagement_rate', 'avg_like_rate', 'avg_comment_rate', 'avg_subs_per_1k_views', 'data_type', 'video_id', 'title', 'channel', 'published_at', 'duration_seconds', 'views', 'likes', 'comments', 'subscribers', 'engagement_rate', 'like_rate', 'comment_rate', 'subs_per_1k_views', 'sentiment', 'positive_probability', 'topic_name_vn', 'topic_id', 'duration_group', 'is_viral']


# TRỰC QUAN HOÁ DỮ LIỆU

Phần này chúng em trực quan bằng power bi ạ